# 4. SFT Data Ablation

Allen AI releases Tulu SFT checkpoints trained without specific data subsets. Same base, same architecture, different SFT data mixtures. Isolates the contribution of safety data to ego-stage displacement.

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *

theme_malign = theme_minimal() + theme(
    figure_size=(10, 6),
    plot_background=element_rect(fill='white'),
    panel_grid_minor=element_blank(),
    text=element_text(family='serif'),
    plot_title=element_text(size=14, weight='bold'),
    plot_subtitle=element_text(size=11, color='#666'),
)

df = pd.read_csv('data/ablation_results.csv')
df['category'] = df['label'].str.replace(r'_\d+$', '', regex=True)

## Overall displacement by ablation variant

In [ ]:
abl_order = df.groupby('ablation')['js_base_ego'].mean().sort_values(ascending=False).index.tolist()

(ggplot(df, aes(x='ablation', y='js_base_ego'))
 + geom_boxplot(aes(fill='ablation'), alpha=0.7, show_legend=False)
 + geom_jitter(width=0.2, alpha=0.3, size=1)
 + scale_x_discrete(limits=abl_order)
 + scale_fill_manual(values={
     'standard': '#e15759', 'no-safety': '#4e79a7', 
     'no-persona': '#59a14f', 'no-math': '#f28e2b', 'no-wildchat': '#b07aa1'
 })
 + labs(title='SFT displacement by data mixture',
        subtitle='Removing any subset reduces displacement — but instruction-following alone still displaces',
        x='', y='JS divergence (base → ego)')
 + theme_malign
)

## Standard vs no-safety by category

Where does removing safety data make the biggest difference?

In [ ]:
compare = df[df.ablation.isin(['standard', 'no-safety'])].copy()
cat_order = compare.groupby('category')['js_base_ego'].mean().sort_values(ascending=False).index.tolist()

(ggplot(compare, aes(x='category', y='js_base_ego', fill='ablation'))
 + geom_col(stat='summary', fun_y=np.mean, position='dodge', alpha=0.85)
 + scale_x_discrete(limits=cat_order)
 + scale_fill_manual(values={'standard': '#e15759', 'no-safety': '#4e79a7'},
                     labels={'standard': 'Standard SFT', 'no-safety': 'No safety data'})
 + labs(title='Safety data effect on SFT displacement',
        subtitle='Removing safety data reduces sexual/power displacement most, but instruction-following still displaces',
        x='', y='JS divergence (base → ego)', fill='')
 + theme_malign
 + theme(axis_text_x=element_text(rotation=45, ha='right'), figure_size=(12, 6))
)